# Setting Up The Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade \
    transformers \
    huggingface_hub \
    datasets \
    sentencepiece \
    evaluate \
    rouge_score \
    accelerate \ peft \ bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 24.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7b7757d657f375dced5c25507ba3b11a5c300cf336001aeebe0a772b85f558d0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: t

In [ ]:
!pip install pandas

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torch

import json
from datasets import Dataset

# TRAINING

In [ ]:
filename = '/content/drive/MyDrive/combinedn_renumbered.ndjson'
from transformers import EarlyStoppingCallback
data = []
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

# Create HuggingFace dataset
texts = [d["text"] for d in data]
heads = [d["headline"] for d in data]

dataset = Dataset.from_dict({"text": texts, "headline": heads})
dataset = dataset.shuffle(seed=42)

# --- Dataset Splitting ---
# Splitting the dataset into 90% Training and 10% Evaluation
print("Total samples:", len(dataset))
train_test_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print("Training samples:", len(train_dataset))
print("Evaluation samples:", len(eval_dataset))

import math

batch_size = 1
grad_accum = 16
train_examples = len(train_dataset)

steps_per_epoch = math.ceil(train_examples / (batch_size * grad_accum))
print("steps_per_epoch:", steps_per_epoch)

# --- END: Dataset Splitting ---

# tokenizer and model initialization
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")
tokenizer.src_lang = "ur_PK"
tokenizer.tgt_lang = "ur_PK"

# model 8-bit
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = MBartForConditionalGeneration.from_pretrained(
    "facebook/mbart-large-50",
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)

def tokenize(batch):
    # Truncate input (text) at 512 tokens, pad to max_length
    inputs = tokenizer(batch["text"], max_length=512, truncation=True, padding="max_length")
    # Truncate labels (headlines) at a shorter max_length (e.g., 64 for headlines)
    labels = tokenizer(batch["headline"], max_length=64, truncation=True, padding="max_length")

    # Important: Replace padding tokens in the labels with -100 so they are ignored in the loss calculation
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    inputs["labels"] = labels["input_ids"]
    return inputs

# Apply tokenization to the split datasets
tokenized_train_ds = train_dataset.map(tokenize, batched=True)
tokenized_eval_ds = eval_dataset.map(tokenize, batched=True)
# --- END: Updated Tokenization Function ---


Total samples: 25017
Training samples: 22515
Evaluation samples: 2502
steps_per_epoch: 1408


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Map:   0%|          | 0/22515 [00:00<?, ? examples/s]

Map:   0%|          | 0/2502 [00:00<?, ? examples/s]

In [ ]:
from transformers import IntervalStrategy, TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    num_train_epochs=20,
    lr_scheduler_type="linear",
    logging_steps=50,

    # *** CORRECTED LINE 1: Argument name changed to 'eval_strategy' ***
    eval_strategy=IntervalStrategy.STEPS,
    eval_steps=steps_per_epoch,

    # *** CORRECTED LINE 2: Argument name is 'save_strategy', value uses IntervalStrategy ***
    save_strategy=IntervalStrategy.STEPS,
    save_steps=steps_per_epoch,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_eval_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

# --- END: Updated Training Arguments and Trainer ---

trainer.train()
trainer.save_model("/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed/final_checkpoint")
tokenizer.save_pretrained("/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed/final_checkpoint")


# EVALUATION

Rogue Scores


In [ ]:
!pip install evaluate rouge-score

In [ ]:
import json
from datasets import Dataset

# ------------------------------
# Load dataset
# ------------------------------
filename = '/content/drive/MyDrive/combinedn_renumbered.ndjson'

data = [json.loads(line) for line in open(filename, "r", encoding="utf-8")]

texts = [d["text"] for d in data]
heads = [d["headline"] for d in data]

dataset = Dataset.from_dict({"text": texts, "headline": heads})
dataset = dataset.shuffle(seed=42)

train_test_split = dataset.train_test_split(test_size=0.1, seed=42)
eval_dataset = train_test_split["test"]

print("Evaluation size:", len(eval_dataset))


# ------------------------------
# Load model + LoRA checkpoint
# ------------------------------
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from peft import PeftModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = "facebook/mbart-large-50"
ckpt = "/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed/checkpoint-1408"

tokenizer = MBart50TokenizerFast.from_pretrained(base_model)

model = MBartForConditionalGeneration.from_pretrained(base_model)
model = PeftModel.from_pretrained(model, ckpt)
model.to(device)
model.eval()


# ------------------------------
# Tokenize only inputs (NO labels)
# ------------------------------
def tokenize_for_eval(batch):
    return tokenizer(
        batch["text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

tokenized_eval = eval_dataset.map(tokenize_for_eval, batched=True)


# ------------------------------
# Run generation
# ------------------------------
preds = []
refs = []

for item in tokenized_eval:
    input_ids = torch.tensor(item["input_ids"]).unsqueeze(0).to(device)
    attention_mask = torch.tensor(item["attention_mask"]).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=64,
            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5
        )

    # decode prediction
    pred = tokenizer.decode(output[0], skip_special_tokens=True)

    # reference (original gold headline)
    ref = item["headline"]

    preds.append(pred)
    refs.append(ref)


# ------------------------------
# Compute ROUGE
# ------------------------------
import evaluate
rouge = evaluate.load("rouge")

results = rouge.compute(predictions=preds, references=refs)
print(results)
#save results
jsonl_path = "/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed/test_result"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for text, ref, pred in zip(eval_dataset["text"], refs, preds):
        f.write(json.dumps({
            "text": text,
            "reference": ref,
            "prediction": pred
        }, ensure_ascii=False) + "\n")

print("Saved JSONL:", jsonl_path)



Evaluation size: 2502


Map:   0%|          | 0/2502 [00:00<?, ? examples/s]

{'rouge1': np.float64(0.144205587910624), 'rouge2': np.float64(0.019491073807620565), 'rougeL': np.float64(0.14368314871912002), 'rougeLsum': np.float64(0.14412565185946485)}


IsADirectoryError: [Errno 21] Is a directory: '/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed'

In [ ]:
#save results
jsonl_path = "/content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed/test_result"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for text, ref, pred in zip(eval_dataset["text"], refs, preds):
        f.write(json.dumps({
            "text": text,
            "reference": ref,
            "prediction": pred
        }, ensure_ascii=False) + "\n")

print("Saved JSONL:", jsonl_path)

Saved JSONL: /content/drive/MyDrive/muskan/mbart-urdu-lora/n_fixed/test_result


In [ ]:
!pip install evaluate bert_score
bertscore = evaluate.load("bertscore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


Bert scores

In [ ]:

import torch
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from peft import PeftModel
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"

# # Eval dataset already loaded as eval_dataset
# # Tokenize only inputs
# def tokenize_for_eval(batch):
#     return tokenizer(batch["text"], max_length=512, truncation=True, padding="max_length")
# tokenized_eval = eval_dataset.map(tokenize_for_eval, batched=True)

# BERTScore evaluation
bertscore = evaluate.load("bertscore")
all_preds = []
all_refs = []
batch_size = 8  # adjust for memory

for i in range(0, len(tokenized_eval), batch_size):
    batch = tokenized_eval.select(range(i, min(i+batch_size, len(tokenized_eval)))).to_dict()
    texts = batch["text"]
    refs = batch["headline"]

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)

    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_preds.extend(decoded_preds)
    all_refs.extend(refs)

results = bertscore.compute(predictions=all_preds, references=all_refs, lang="ur")
print("BERTScore:", results)



tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore: {'precision': [0.6471728086471558, 0.9217442274093628, 0.9013381004333496, 0.7451561093330383, 0.7487063407897949, 0.7913804054260254, 0.8029106855392456, 0.8631898164749146, 0.7494881749153137, 0.8929282426834106, 0.6960935592651367, 0.7303279638290405, 0.8166242837905884, 0.9378953576087952, 0.9062446355819702, 0.8683795928955078, 0.9632222652435303, 0.869981586933136, 0.8305933475494385, 0.9613944292068481, 0.922799825668335, 0.8794326782226562, 0.7531386613845825, 0.8115752935409546, 0.825559139251709, 0.9873727560043335, 0.8519562482833862, 0.8572080135345459, 0.7345755100250244, 0.7437740564346313, 0.7996739149093628, 0.9094510674476624, 0.814456582069397, 0.8188422918319702, 0.8116182088851929, 0.7098439931869507, 0.8950674533843994, 0.898119330406189, 0.6891075372695923, 0.7525275349617004, 0.9098238945007324, 0.9007788300514221, 0.9564957618713379, 0.6567590236663818, 0.9158222675323486, 0.6483092904090881, 0.7676296234130859, 0.8626928925514221, 0.8307839035987854,

In [ ]:
precision = sum(results["precision"]) / len(results["precision"])
recall    = sum(results["recall"])    / len(results["recall"])
f1        = sum(results["f1"])        / len(results["f1"])

print("\n=== BERTScore ===")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")



=== BERTScore ===
Precision: 0.8181
Recall:    0.8099
F1 Score:  0.8133


Bleu Scores

In [ ]:
!pip install sacrebleu evaluate nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from peft import PeftModel
import evaluate
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Compute SacreBLEU
# -------------------------
refs_for_sacre = [all_refs]
bleu = sacrebleu.corpus_bleu(all_preds, refs_for_sacre, smooth_method='exp')

print("\n=== SacreBLEU ===")
print("BLEU-4 score (percent): {:.2f}".format(bleu.score))
print("BLEU-4 score (0–1): {:.4f}".format(bleu.score / 100.0))




=== SacreBLEU ===
BLEU-4 score (percent): 29.47
BLEU-4 score (0–1): 0.2947
